# Module 9: Performance & Cost Engineering for AI Systems

Build a caching decorator and profiling workflow to cut latency and spend.


## 1. Setup

Load performance/cost core utilities.


In [ ]:
import sys
from pathlib import Path

module_dir = Path.cwd()
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

from perf_cost_core import (
    SemanticCache,
    LLMServiceSimulator,
    caching_decorator,
    batch_infer,
    generate_query_load,
    profile_prompt_optimization,
)

print("✓ Setup complete")


## 2. Baseline Query Cost and Latency


In [ ]:
service = LLMServiceSimulator()
baseline = service.answer("What is the daily per diem?")
baseline


## 3. Semantic Caching Decorator

Repeated/near-duplicate queries should return instantly with zero API cost.


In [ ]:
cache = SemanticCache(similarity_threshold=0.82)
cached_call = caching_decorator(cache, service)

queries = [
    "What is the daily per diem?",
    "what is daily per diem",
    "What is the meal limit?",
    "what is the meal limit please",
    "What is the daily per diem",
]

run_metrics = []
for q in queries:
    ans, m = cached_call(q)
    run_metrics.append(m.to_dict())

run_metrics


## 4. Load Test with Repeated Traffic

Simulate many users asking repetitive policy questions.


In [ ]:
load_queries = generate_query_load(n=120, seed=42)

hits = 0
misses = 0
total_cost = 0.0
for q in load_queries:
    _, m = cached_call(q)
    if m.cached:
        hits += 1
    else:
        misses += 1
    total_cost += float(m.estimated_cost_usd)

{
    "queries": len(load_queries),
    "hits": hits,
    "misses": misses,
    "cache_hit_rate": round(hits / len(load_queries), 4),
    "total_cost_usd": round(total_cost, 4),
}


## 5. Batch Optimization

Compare single-call processing vs batch processing.


In [ ]:
batch_stats = batch_infer(load_queries[:60], batch_size=10)
batch_stats


## 6. Prompt Token Optimization

Shorter system prompts reduce token and cost footprint.


In [ ]:
prompt_stats = profile_prompt_optimization("What is the daily per diem?")
prompt_stats


## 7. Assertions (Smoke Checks)


In [ ]:
assert hits > misses
assert batch_stats["speedup_x"] > 1.0
assert int(prompt_stats["token_reduction"]) > 0

print("✓ Module 9 performance/cost smoke checks passed")
